In [1]:
import xarray as xr
import torch
import yaml
import sys
from pathlib import Path
root_dir = Path.cwd().parent   
sys.path.append(str(root_dir))

from data.dataset import *
from data.dataloader import *
from data.preprocessing import *

print("Torch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("CUDA version (runtime):", torch.version.cuda)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU only")

Torch version: 2.3.1
CUDA available: True
CUDA version (runtime): 12.1
GPU: Quadro T1000 with Max-Q Design


In [ ]:
# Load config
config_path = Path.cwd().parent / "utils" / "default_config.yaml"
with open(config_path, "r") as f:
    config = yaml.safe_load(f)

data_cfg = config["data"]
train_cfg = config["training"]
print("Config loaded!")

# Load full dataset
url = data_cfg["dataset_url"]
ds = load_full_dataset(url)

# Reduce dataset using config
reduced_ds = reduce_dataset(ds, data_cfg)
print(reduced_ds)

Config loaded!
Opening dataset from: gs://weatherbench2/datasets/era5_daily/1959-2023_01_10-full_37-1h-0p25deg-chunk-1-s2s.zarr


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

variables_to_keep = data_cfg["variables_to_keep"]
target_var = data_cfg["target_variable"]

train_loader, val_loader, test_loader = get_dataloaders(
    reduced_ds, variables_to_keep, target_var,
    input_length=7, forecast_horizon=1,
    batch_size=8, num_workers=2,
    device=device
)
print("\nDataloaders ready!")

Using device: cuda

Dataloaders ready!
